In [0]:
from pyspark.sql.functions import regexp_extract, col, when, nullif, regexp_extract, lit
from pyspark.sql.types import IntegerType

In [0]:
silver_conta = spark.read.table('mobills.silver.contas')
silver_cartao = spark.read.table('mobills.silver.cartoes')
silver_transacoes = spark.read.table('mobills.silver.transacoes')
silver_categorias = spark.read.table('mobills.silver.categorias')

regex = r"\((\d+)/(\d+)\)"

df_gold_raw = (
    silver_transacoes.alias('trs')
    .join(silver_conta.alias('ct'), on=(col('trs.conta_id') == col('ct.id')))
    .join(silver_cartao.alias('crt'), on=(col('trs.cartao_id') == col('crt.id')), how='left')
    .join(silver_categorias.alias('cat'), on=(col('trs.tipo_transacao_filho_id') == col('cat.id')), how='left')
    .withColumn(
        "parcela_atual",
        nullif(regexp_extract("trs.descricao", regex, 1), lit("")).cast("int")
    )
    .withColumn(
        "total_parcelas",
        nullif(regexp_extract("trs.descricao", regex, 2), lit("")).cast("int")
    )
    .withColumn(
        "futuro_vendido",
        (col("total_parcelas") - col("parcela_atual")) * col("valor_abs")
    )
    .select(
        'trs.unique_id', 'trs.id', 'trs.data', 'trs.descricao', 'trs.tipo', 'trs.situacao', col('ct.nome').alias('conta'), col('crt.nome').alias('cartao'), 
        'cat.natureza', col('cat.categoriaPai').alias('categoria'), col('cat.categoria').alias('subcategoria'), 'trs.valor', 'trs.valor_abs',
        'parcela_atual', 'total_parcelas', 'futuro_vendido'
    )
)

(
    df_gold_raw
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable("mobills.gold.transacoes")
)